# مختبر: هندسة الميزات (Feature Engineering)

في هذا المختبر سنطبق الخطوات الأساسية لهندسة الميزات باستخدام Python: معالجة القيم المفقودة، ترميز الميزات الفئوية، وقياس الميزات. سننتبه بشكل خاص لمشكلة تسرّب بيانات الاختبار (Data Leakage) ونظهر أثرها.

المكتبات: `numpy`, `pandas`, `matplotlib`, `sklearn`.

## الخطوة 1: إنشاء بيانات اصطناعية

سننشئ مجموعة بيانات تحتوي على ميزات رقمية ('age', 'income') وفئوية ('gender', 'education') مع بعض القيم المفقودة، وهدف ثنائي 'purchase'.

In [ ]:
import numpy as np
import pandas as pd
np.random.seed(42)

n = 200
# إنشاء الميزات:
age = np.random.normal(30, 10, n)          # TODO: املأ
income = np.random.lognormal(1, 0.5, n) * 1000  # TODO: املأ
gender = np.random.choice(['male', 'female'], n)  # TODO: املأ
education = np.random.choice(['high school', 'bachelor', 'master', 'PhD'], n, p=[0.3,0.4,0.2,0.1])  # TODO: املأ

# إدخال قيم مفقودة:
age[np.random.choice(n, int(0.2*n), replace=False)] = np.nan  # TODO: املأ
gender[np.random.choice(n, int(0.1*n), replace=False)] = np.nan  # TODO: املأ

# الهدف purchase:
purchase = ((age + income/1000) / 50 + np.random.randn(n)*0.5) > 0.7  # TODO: حول إلى int
purchase = purchase.astype(int)  # TODO: حول إلى int

df = pd.DataFrame({'age': age, 'income': income, 'gender': gender, 'education': education, 'purchase': purchase})
df.head()

## الخطوة 2: تقسيم البيانات إلى مجموعتي تدريب واختبار

سنفصل الميزات عن الهدف، ثم نقسّم باستخدام `train_test_split` مع `test_size=0.2` و `random_state=42`. من المهم إجراء التقسيم قبل أي معالجة لتجنب التسرب.

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop('purchase', axis=1)
y = df['purchase']

# TODO: استخدم train_test_split لتقسيم X, y مع test_size=0.2 و random_state=42
X_train, X_test, y_train, y_test = ...

print(X_train.shape, X_test.shape)

## الخطوة 3: استكشاف القيم المفقودة في مجموعة التدريب

لنفحص نسبة القيم المفقودة في كل عمود من أعمدة مجموعة التدريب فقط (لا نطلع على الاختبار).

In [ ]:
# حساب النسب المئوية للقيم المفقودة في X_train
missing_perc = ...  # TODO: احسب (عدد القيم المفقودة / إجمالي الصفوف) * 100
print(missing_perc)

## الخطوة 4: معالجة القيم المفقودة (Imputation)

سنستخدم `SimpleImputer` من `sklearn.impute` لملء القيم المفقودة. التعامل مع الميزات الرقمية باستخدام المتوسط (mean)، ومع الفئوية باستخدام القيمة الأكثر تكرارًا (most_frequent). سنُنشئ imputer منفصل لكل نوع. من المهم أن نُطبق `fit` على مجموعة التدريب فقط، ثم `transform` على التدريب والاختبار.

In [ ]:
from sklearn.impute import SimpleImputer

num_cols = ['age', 'income']
cat_cols = ['gender', 'education']

# TODO: أنشئ imputer_num بمتوسط (strategy='mean')
imputer_num = ...
# TODO: استخدم fit_transform على X_train[num_cols] و transform على X_test[num_cols]
X_train_num_imp = imputer_num...
X_test_num_imp = imputer_num...

# TODO: أنشئ imputer_cat بالقيمة الأكثر تكرارًا (strategy='most_frequent')
imputer_cat = ...
# TODO: استخدم fit_transform على X_train[cat_cols] و transform على X_test[cat_cols]
X_train_cat_imp = imputer_cat...
X_test_cat_imp = imputer_cat...

# TODO: حوّل المصفوفات إلى DataFrames مع الأعمدة الصحيحة
X_train_num_imp = pd.DataFrame(..., columns=num_cols, index=X_train.index)
X_test_num_imp = pd.DataFrame(..., columns=num_cols, index=X_test.index)
X_train_cat_imp = pd.DataFrame(..., columns=cat_cols, index=X_train.index)
X_test_cat_imp = pd.DataFrame(..., columns=cat_cols, index=X_test.index)

# دمج الأعمدة
X_train_imp = pd.concat([X_train_num_imp, X_train_cat_imp], axis=1)
X_test_imp = pd.concat([X_test_num_imp, X_test_cat_imp], axis=1)

print('القيم المفقودة بعد المعالجة:')
print(X_train_imp.isnull().sum())

## الخطوة 5: ترميز الميزات الفئوية (Categorical Encoding)

الميزات الفئوية لا تزال نصوصًا. سنستخدم `OneHotEncoder` لتحويلها إلى أعمدة ثنائية. كالعادة، `fit` على التدريب، ثم `transform` على التدريب والاختبار.

In [ ]:
from sklearn.preprocessing import OneHotEncoder

# TODO: أنشئ OneHotEncoder مع sparse=False و handle_unknown='ignore'
encoder = ...

# TODO: طبق fit_transform على X_train_imp[cat_cols] و transform على X_test_imp[cat_cols]
X_train_cat_enc = encoder...
X_test_cat_enc = encoder...

# حوّل إلى DataFrame بأسماء الأعمدة الجديدة التي يعيدها encoder.get_feature_names(cat_cols)
X_train_cat_enc = pd.DataFrame(..., columns=encoder.get_feature_names(cat_cols), index=X_train_imp.index)
X_test_cat_enc = pd.DataFrame(..., columns=encoder.get_feature_names(cat_cols), index=X_test_imp.index)

# دمج مع الأعمدة العددية لإنشاء X_train_processed النهائية
X_train_processed = pd.concat([X_train_imp[num_cols], X_train_cat_enc], axis=1)
X_test_processed = pd.concat([X_test_imp[num_cols], X_test_cat_enc], axis=1)

print('شكل البيانات بعد الترميز:', X_train_processed.shape)

## الخطوة 6: قياس الميزات (Scaling)

الميزات الرقمية ('age', 'income') لها مقاييس مختلفة. سنستخدم `StandardScaler` لتوحيدها بحيث تكون بمتوسط 0 وانحراف 1. نطبّق `fit` على التدريب فقط، ثم `transform` على التدريب والاختبار.

In [ ]:
from sklearn.preprocessing import StandardScaler

# TODO: أنشئ scaler من StandardScaler
scaler = ...

# TODO: طبق fit_transform على X_train_processed[num_cols] و transform على X_test_processed[num_cols]
X_train_scaled = scaler...
X_test_scaled = scaler...

# حوّل إلى DataFrame
X_train_scaled = pd.DataFrame(..., columns=num_cols, index=X_train_processed.index)
X_test_scaled = pd.DataFrame(..., columns=num_cols, index=X_test_processed.index)

# دمج الأعمدة المقاسة مع الأعمدة المشفرة الأخرى
other_cols = [col for col in X_train_processed.columns if col not in num_cols]
X_train_final = pd.concat([X_train_scaled, X_train_processed[other_cols]], axis=1)
X_test_final = pd.concat([X_test_scaled, X_test_processed[other_cols]], axis=1)

print('إحصائيات (mean/std) بعد القياس:')
print(X_train_scaled.describe().loc[['mean','std']])

## الخطوة 7: تسرّب البيانات (Data Leakage) – مثال خطر

لنُظهر ما يحدث إذا قمنا بقياس عمود 'income' على البيانات الكاملة (قبل التقسيم) وقارنّا ذلك بالقياس الصحيح على التدريب فقط.

In [ ]:
scaler_full = StandardScaler()
scaler_full.fit(df[['income']])  # تسرب: استخدام كامل البيانات
print('المتوسط (كامل البيانات):', scaler_full.mean_[0])
print('الانحراف (كامل البيانات):', scaler_full.scale_[0])

# TODO: اطبع متوسط وانحراف الدخل من scaler الذي تعلم على التدريب فقط (لاحظ أن العمود الثاني في num_cols هو income)
print('المتوسط (التدريب فقط):', scaler.mean_[1])
print('الانحراف (التدريب فقط):', scaler.scale_[1])

## الخلاصة

في هذا المختبر طبقنا خطوات أساسية لهندسة الميزات مع الالتزام بمبدأ منع تسرّب البيانات: تعاملنا مع القيم المفقودة باستخدام imputation، ثم الترميز الفئوي، ثم قياس الميزات، مع تطبيق `fit` على بيانات التدريب فقط. رأينا كيف أن دمج بيانات الاختبار في حساب معاملات القياس (أو أي معالجة أخرى) قد يؤدي إلى تقديرات متفائلة جدًا لأداء النموذج.